In [ ]:
!nvidia-smi

import glob, pathlib
wheel_files = glob.glob('/kaggle/input/datasets/fabioveroli/minkowskiengine-p100/*.whl')
if wheel_files:
    print(f"\u2705 Wheel trovato: {wheel_files[0]}")
else:
    print("\u274c Wheel NON trovato in /kaggle/input/datasets/fabioveroli/minkowskiengine-p100/")
    print("   Aggiungere il Dataset 'minkowski-wheel-p100' come Input del notebook")


In [ ]:
%cd /kaggle/working

# PyTorch 2.2.0 + CUDA 12.1 (versione richiesta da icg_net)
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q

# Sistema + numpy/scipy compatibili
!apt-get install -y libopenblas-dev -q
!pip install "numpy<2" "scipy==1.13.1" -q

print("\u2705 PyTorch e dipendenze base installate")
import torch
print(f"   torch: {torch.__version__}, CUDA disponibile: {torch.cuda.is_available()}")


In [ ]:
import glob, subprocess

# Carica il wheel pre-compilato dal Dataset Kaggle 'minkowski-wheel-p100'
# invece di ricompilare da sorgente (~5 min su P100)
wheel_files = glob.glob('/kaggle/input/datasets/fabioveroli/minkowskiengine-p100/*.whl')
assert wheel_files, "\u274c Wheel non trovato. Aggiungi il Dataset come Input."

wheel_path = wheel_files[0]
print(f"Installo: {wheel_path}")
result = subprocess.run(['pip', 'install', wheel_path, '-q'], capture_output=True, text=True)

if result.returncode != 0:
    print(f"\u274c Installazione fallita:\n{result.stderr[-500:]}")
else:
    import MinkowskiEngine as ME
    print(f"\u2705 MinkowskiEngine {ME.__version__} installato dal wheel pre-compilato")


In [ ]:
%cd /kaggle/working

# Repo principale e benchmark
!git clone https://github.com/renezurbruegg/icg_net.git
!git clone https://github.com/renezurbruegg/icg_benchmark.git

# Dipendenze PyG (esatte per torch 2.2.0+cu121)
!pip install torch_geometric -q
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-2.2.0+cu121.html -q

# Dipendenze generali
!pip install hydra-core open3d trimesh networkx einops -q

print("\u2705 Dipendenze Python installate")


In [ ]:
import subprocess, pathlib

icgnet_py = pathlib.Path("/kaggle/working/icg_net/icg_net/icg_net.py")
text = icgnet_py.read_text()

text = text.replace(
    "from hydra.experimental import compose, initialize",
    "from hydra import compose, initialize"
)
text = text.replace(
    "with initialize():",
    "with initialize(version_base=None):"
)
text = text.replace(
    "with initialize(config_path",
    "with initialize(version_base=None, config_path"
)

icgnet_py.write_text(text)
print("\u2705 Patch Hydra applicata")


In [ ]:
%cd /kaggle/working/icg_net/icg_net/third_party/pointnet2
!pip install . -q
print("\u2705 PointNet2 installato")


In [ ]:
%cd /kaggle/working/icg_net/icg_net/utils/mcubes

import os, subprocess, pathlib

setup_text = pathlib.Path("setup.py").read_text()

cpp_exists = pathlib.Path("src/mcubes.cpp").exists()

if cpp_exists:
    setup_text = setup_text.replace('"src/mcubes.pyx"', '"src/mcubes.cpp"')
    setup_text = setup_text.replace("cythonize(ext_modules)", "ext_modules")
    setup_text = setup_text.replace("from Cython.Build import cythonize\n", "")
    pathlib.Path("setup.py").write_text(setup_text)
    print("✅ Patch mcubes: uso del .cpp pre-compilato")
else:
    subprocess.run(["pip", "install", "cython", "-q"], check=True)
    print("✅ Cython installato per compilare mcubes.pyx")

!python setup.py build_ext --inplace 2>&1 | tail -10
print("✅ mcubes compilato")


In [ ]:
import sys, subprocess

print("\U0001f504 Aggiorno numpy a 2.x (icg_net lo richiede)...")
!pip install "numpy==2.0.2" -q
!pip install loguru -q
print("\u2705 numpy aggiornato")

try:
    import MinkowskiEngine as ME
    print(f"\u2705 MinkowskiEngine ancora funzionante: {ME.__version__}")
except ImportError as e:
    print(f"\u26a0\ufe0f  MinkowskiEngine non importabile: {e}")
    print("   Reinstalla il .whl se necessario")

ICG_NET_PATH = '/kaggle/working/icg_net'
if ICG_NET_PATH not in sys.path:
    sys.path.insert(0, ICG_NET_PATH)

try:
    import icg_net
    print(f"\u2705 icg_net importato: {icg_net.__file__}")
except ImportError as e:
    print(f"\u274c icg_net: {e}")

result = subprocess.run(
    ["pip", "install", "-e", "/kaggle/working/icg_benchmark", "-q"],
    capture_output=True, text=True
)
print("\u2705 icg_benchmark OK" if result.returncode == 0 else f"\u274c {result.stderr[-300:]}")


In [ ]:
%cd /kaggle/working/icg_benchmark
!python scripts/download_data.py

import subprocess, pathlib

result = subprocess.run(
    ['find', '/kaggle/working/icg_benchmark', '-name', 'config.yaml'],
    capture_output=True, text=True
)
print("Config trovati:\n", result.stdout)

configs = [p for p in result.stdout.strip().split('\n') if p]
if configs:
    CONFIG_PATH = configs[0]
    print(f"\u2705 Usando: {CONFIG_PATH}")
else:
    result2 = subprocess.run(
        ['find', '/kaggle/working/icg_net', '-name', 'config.yaml'],
        capture_output=True, text=True
    )
    configs = [p for p in result2.stdout.strip().split('\n') if p]
    CONFIG_PATH = configs[0] if configs else None
    print(f"CONFIG_PATH = {CONFIG_PATH}")


In [ ]:
import pathlib, os

icgnet_py = pathlib.Path("/kaggle/working/icg_net/icg_net/icg_net.py")
text = icgnet_py.read_text()

old_block = '''        if isinstance(config, str):
            with initialize(version_base=None):
                cfg = compose(config_name=config, return_hydra_config=True, overrides=[])'''
new_block = '''        if isinstance(config, str):
            import os as _os
            from hydra import initialize_config_dir
            _config_dir = _os.path.dirname(_os.path.abspath(config))
            _config_name = _os.path.splitext(_os.path.basename(config))[0]
            with initialize_config_dir(version_base=None, config_dir=_config_dir):
                cfg = compose(config_name=_config_name, return_hydra_config=True, overrides=[])'''

if old_block in text:
    text = text.replace(old_block, new_block)
    icgnet_py.write_text(text)
    print("\u2705 Patch Hydra initialize_config_dir applicata")
else:
    print("\u26a0\ufe0f  Blocco non trovato — mostra le righe con initialize/compose:")
    for i, line in enumerate(text.splitlines()):
        if 'initialize' in line or 'compose' in line:
            print(f"  {i+1}: {line}")


In [ ]:
import pathlib, re, subprocess

CONFIG_DIR = "/kaggle/working/icg_benchmark/data/icgnet/51--0.656"

result = subprocess.run(
    ['find', '/kaggle/working/icg_benchmark', '-name', '*.ckpt'],
    capture_output=True, text=True
)
print("Checkpoint trovati:", result.stdout)
CKPT_PATH = result.stdout.strip().split('\n')[0]
print(f"Uso: {CKPT_PATH}")

config_yaml = pathlib.Path(f"{CONFIG_DIR}/config.yaml")
text = config_yaml.read_text()

text_patched = re.sub(
    r'checkpoint:\s*.*checkpoint\.ckpt',
    f'checkpoint: {CKPT_PATH}',
    text
)

if text_patched != text:
    config_yaml.write_text(text_patched)
    print(f"\u2705 config.yaml patchato con: {CKPT_PATH}")
else:
    print("\u26a0\ufe0f  Pattern non trovato. Righe con 'checkpoint':")
    for line in text.splitlines():
        if 'checkpoint' in line.lower():
            print(f"  {line}")


In [ ]:
import sys, torch

for mod in list(sys.modules.keys()):
    if 'icg_net' in mod:
        del sys.modules[mod]

if '/kaggle/working/icg_net' not in sys.path:
    sys.path.insert(0, '/kaggle/working/icg_net')

from icg_net import ICGNetModule, get_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model: ICGNetModule = get_model(CONFIG_PATH, device=device)
print("\u2705 Modello caricato!")


In [ ]:
# Introspect struttura output su pointcloud sintetica (debug)
import torch, numpy as np

N = 100
pts_t = torch.randn(N, 3, device=device).float() * 0.1
nrm_t = torch.nn.functional.normalize(pts_t, dim=1)

with torch.no_grad():
    _out = model(pts_t, normals=nrm_t, grasp_pts=pts_t, grasp_normals=nrm_t,
                 n_grasps=10, each_object=True, return_meshes=False, return_scene_grasps=True)

print("Tipo output:", type(_out))
print("Attributi pubblici:", [a for a in dir(_out) if not a.startswith('_')])
print("scene_grasp_poses len:", len(_out.scene_grasp_poses))
for i, sp in enumerate(_out.scene_grasp_poses):
    if sp is not None and hasattr(sp, 'shape'):
        print(f"  [{i}] shape={tuple(sp.shape)}, dtype={sp.dtype}")
    elif sp is not None:
        print(f"  [{i}] type={type(sp)}")
    else:
        print(f"  [{i}] None")


In [ ]:
import open3d as o3d, json, pathlib, torch, numpy as np
from scipy.spatial.transform import Rotation

# ── Configurazione ────────────────────────────────────────────────────────────
INPUT_DIR = "/kaggle/input/icgnet-scenes"   # Dataset Kaggle con i file .pcd
N_GRASPS  = 64                              # Numero di grasp per scena
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = pathlib.Path("/kaggle/working/grasps")
OUTPUT_DIR.mkdir(exist_ok=True)

pcd_files = sorted(pathlib.Path(INPUT_DIR).glob("*.pcd"))
print(f"Scene trovate: {len(pcd_files)}")

for pcd_file in pcd_files:
    scene_id = pcd_file.stem
    print(f"\n\u2192 {scene_id} ...")

    pcd = o3d.io.read_point_cloud(str(pcd_file))
    print(f"   {len(pcd.points)} punti")

    if len(pcd.points) < 10:
        print("   \u26a0\ufe0f  Cloud troppo piccola, skip")
        continue

    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
    )
    pcd.orient_normals_towards_camera_location([0, 0, 2])

    pts = np.asarray(pcd.points, dtype=np.float32)
    nrm = np.asarray(pcd.normals, dtype=np.float32)
    pts_t = torch.from_numpy(pts).to(device)
    nrm_t = torch.from_numpy(nrm).to(device)

    with torch.no_grad():
        out = model(pts_t, normals=nrm_t, grasp_pts=pts_t, grasp_normals=nrm_t,
                    n_grasps=N_GRASPS, each_object=True,
                    return_meshes=False, return_scene_grasps=True)

    # scene_grasp_poses: [rot(G,3,3), centers(G,3), scores(G,), widths(G,), instance_ids(G,)]
    rot_matrices = out.scene_grasp_poses[0].cpu().numpy()
    centers      = out.scene_grasp_poses[1].cpu().numpy()
    scores       = out.scene_grasp_poses[2].cpu().numpy()
    widths       = out.scene_grasp_poses[3].cpu().numpy()
    inst_ids     = out.scene_grasp_poses[4].cpu().numpy()
    G = len(centers)

    # Semantic class per istanza (class_predictions)
    if hasattr(out, 'class_predictions') and out.class_predictions is not None:
        cls = out.class_predictions.cpu().numpy().flatten()
    else:
        cls = np.zeros(G, dtype=int)

    grasps = []
    for i in range(G):
        quat = Rotation.from_matrix(rot_matrices[i]).as_quat()   # [qx, qy, qz, qw]
        grasps.append({
            "pose": {
                "position": centers[i].tolist(),
                "quaternion": quat.tolist(),
            },
            "score":          float(scores[i]),
            "width":          float(widths[i]),
            "instance_id":    int(inst_ids[i]),
            "semantic_class": int(cls[i]) if i < len(cls) else 0,
        })

    out_path = OUTPUT_DIR / f"{scene_id}.json"
    out_path.write_text(json.dumps({"scene_id": scene_id, "grasps": grasps}, indent=2))
    print(f"   \u2705 {G} grasps \u2192 {out_path}")

print(f"\n\u2705 Done. {len(list(OUTPUT_DIR.glob('*.json')))} file JSON in {OUTPUT_DIR}")


In [ ]:
import json, pathlib

output_files = sorted(pathlib.Path('/kaggle/working/grasps').glob('*.json'))
print(f"File generati: {len(output_files)}")

for f in output_files:
    data = json.loads(f.read_text())
    n = len(data['grasps'])
    score_avg = sum(g['score'] for g in data['grasps']) / max(n, 1)
    print(f"  {f.name}: {n} grasps, score medio {score_avg:.3f}")
    if n > 0:
        g0 = data['grasps'][0]
        print(f"    Esempio: pos={[round(x,3) for x in g0['pose']['position']]}, "
              f"score={g0['score']:.3f}, instance_id={g0['instance_id']}")
